# 🚗 EfficientDet-D2 para Detección de Placas Vehiculares

## Setup inicial
**IMPORTANTE:** Asegúrate de activar GPU en Colab:
- `Runtime` → `Change runtime type` → `Hardware accelerator: GPU` → `GPU type: T4`

---

### Características del entrenamiento:
- ✅ **Mixed Precision (AMP)**: 2x más rápido
- ✅ **Early Stopping**: Detiene si no mejora en 10 épocas
- ✅ **Data Augmentation**: Mejora generalización
- ✅ **Exportación ONNX**: Para producción optimizada
- ✅ **GPU Tesla T4**: 16GB VRAM gratis

In [ ]:
# Verificar GPU disponible
!nvidia-smi

import torch
print(f"\n{'='*60}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️  GPU no detectada. Activa GPU en: Runtime → Change runtime type → GPU")
print(f"{'='*60}")

## 📦 Instalación de dependencias

Instala PyTorch, EfficientDet y herramientas necesarias.

In [ ]:
%%capture
# Instalar PyTorch con CUDA 11.8 (optimizado para Colab T4)
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118

# Instalar EfficientDet y dependencias
!pip install effdet albumentations opencv-python pycocotools tqdm pyyaml

# Para exportación ONNX
!pip install onnx onnxruntime-gpu

print("✅ Dependencias instaladas correctamente")

In [ ]:
# Verificar instalación
import torch
import effdet
import albumentations as A
import cv2
import onnx

print("✅ Todas las librerías importadas correctamente")
print(f"   PyTorch: {torch.__version__}")
print(f"   CUDA disponible: {torch.cuda.is_available()}")

## 📂 Subir y organizar datasets



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Crear estructura de directorios
!mkdir -p datasets

# MODIFICA ESTA RUTA según donde tengas tus datasets en Drive
# Ejemplo: si tus datasets están en "Mi unidad/ML/datasets/"
DRIVE_DATASET_PATH = "/content/drive/MyDrive/datasets"  # ← MODIFICAR AQUÍ

# Copiar datasets desde Drive
!cp -r {DRIVE_DATASET_PATH}/* datasets/

print("\n✅ Datasets copiados desde Google Drive")
print("\n📁 Verificando estructura:")
!ls -R datasets/ | head -30

### Opción 2: Upload directo de archivo ZIP

In [ ]:
from google.colab import files
import zipfile

# Subir archivo ZIP (aparecerá botón para seleccionar archivo)
print("📤 Selecciona tu archivo ZIP con los datasets...")
uploaded = files.upload()

# Extraer
!mkdir -p datasets
for filename in uploaded.keys():
    print(f"\n📦 Extrayendo {filename}...")
    with zipfile.ZipFile(filename, 'r') as zip_ref:
        zip_ref.extractall('datasets/')
    print(f"✅ {filename} extraído")

# Verificar estructura
print("\n📁 Estructura de datasets:")
!ls -R datasets/ | head -30

### Opción 3: Descargar desde URL pública

In [ ]:
# Si tienes tu dataset en una URL pública (Dropbox, Google Drive compartido, etc.)
!mkdir -p datasets

# MODIFICA ESTA URL
DATASET_URL = "https://tu-url-aqui.com/datasets.zip"  # ← MODIFICAR AQUÍ

# Descargar y extraer
!wget -O datasets.zip "{DATASET_URL}"
!unzip -q datasets.zip -d datasets/
!rm datasets.zip

print("✅ Dataset descargado y extraído")
!ls -R datasets/ | head -30

In [ ]:
# Contar imágenes en cada dataset
import os
from pathlib import Path

def count_images(path):
    if not os.path.exists(path):
        return 0
    return len(list(Path(path).glob("*.jpg")))

print("\n📊 Resumen de datasets:")
print("="*60)

# Dataset global
if os.path.exists("datasets/license-plates"):
    train = count_images("datasets/license-plates/train/images")
    valid = count_images("datasets/license-plates/valid/images")
    test = count_images("datasets/license-plates/test/images")
    print(f"Global Dataset:")
    print(f"  Train: {train} imágenes")
    print(f"  Valid: {valid} imágenes")
    print(f"  Test:  {test} imágenes")
    print(f"  Total: {train + valid + test} imágenes\n")

# Datasets Ecuador
ec_base = "datasets/license-plates-ec-combined"
if os.path.exists(ec_base):
    for ec_name in ["license-plates-ec-1", "license-plates-ec-2", "license-plates-ec-4"]:
        ec_path = f"{ec_base}/{ec_name}"
        if os.path.exists(ec_path):
            train = count_images(f"{ec_path}/train/images")
            valid = count_images(f"{ec_path}/valid/images")
            test = count_images(f"{ec_path}/test/images")
            print(f"{ec_name}:")
            print(f"  Train: {train} | Valid: {valid} | Test: {test} | Total: {train+valid+test}\n")

print("="*60)

## Entrenamiento EfficientDet-D2

In [ ]:
import sys
import os
import glob
import yaml
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from effdet import get_efficientdet_config, EfficientDet, DetBenchTrain
from effdet.efficientdet import HeadNet
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import numpy as np
from pathlib import Path
from typing import List, Dict, Tuple
from tqdm import tqdm
import json

# ── Configuración ──────────────────────────────────────────────────────────────
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME   = "tf_efficientdet_d2"
EPOCHS       = 100
IMG_SIZE     = 768
BATCH_SIZE   = 8   # T4 GPU tiene 16GB, puede manejar batch 8-12
LR           = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS  = 2   # Colab tiene 2 CPUs
NUM_CLASSES  = 1
USE_AMP      = True  # Mixed Precision
EXPORT_ONNX  = True

EC_BASE = "datasets/license-plates-ec-combined"

DATASETS = {
    "global": {
        "yaml": "datasets/license-plates/data.yaml",
        "name": "efficientdet_d2_plates_global",
    },
    "ecuador": {
        "yaml": f"{EC_BASE}/license-plates-ec-1/data.yaml",
        "name": "efficientdet_d2_plates_ecuador",
    },
    "ecuador2": {
        "yaml": f"{EC_BASE}/license-plates-ec-2/data.yaml",
        "name": "efficientdet_d2_plates_ecuador2",
    },
    "ecuador4": {
        "yaml": f"{EC_BASE}/license-plates-ec-4/data.yaml",
        "name": "efficientdet_d2_plates_ecuador4",
    },
    "combined": {
        "yaml": f"{EC_BASE}/data_combined.yaml",
        "name": "efficientdet_d2_plates_ec_combined",
    },
    "combined_all": {
        "yaml": f"{EC_BASE}/data_combined_all.yaml",
        "name": "efficientdet_d2_plates_combined_all",
    },
}

print(f"\n{'='*60}")
print(f"  Configuración de entrenamiento")
print(f"  Dispositivo: {DEVICE}")
print(f"  Mixed Precision: {'✅ ENABLED' if USE_AMP else '❌ DISABLED'}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Image size: {IMG_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"{'='*60}")

In [ ]:
# ── Dataset YOLO → COCO ────────────────────────────────────────────────────────
class YOLODataset(Dataset):
    """Convierte dataset YOLO a formato EfficientDet (COCO-style)"""
    
    def __init__(self, image_dirs: List[str], transform=None):
        self.image_paths = []
        self.label_paths = []
        self.transform = transform
        
        for img_dir in (image_dirs if isinstance(image_dirs, list) else [image_dirs]):
            img_dir = Path(img_dir)
            label_dir = img_dir.parent / "labels"
            
            for img_path in img_dir.glob("*.jpg"):
                label_path = label_dir / f"{img_path.stem}.txt"
                if label_path.exists():
                    self.image_paths.append(str(img_path))
                    self.label_paths.append(str(label_path))
        
        print(f"  [dataset] Cargadas {len(self.image_paths)} imágenes")
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w = image.shape[:2]
        
        boxes = []
        labels = []
        
        with open(self.label_paths[idx], 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    cls, xc, yc, bw, bh = map(float, parts)
                    
                    x_min = (xc - bw / 2) * w
                    y_min = (yc - bh / 2) * h
                    x_max = (xc + bw / 2) * w
                    y_max = (yc + bh / 2) * h
                    
                    boxes.append([x_min, y_min, x_max, y_max])
                    labels.append(int(cls))
        
        boxes = np.array(boxes, dtype=np.float32)
        labels = np.array(labels, dtype=np.int64)
        
        if self.transform:
            transformed = self.transform(
                image=image,
                bboxes=boxes,
                labels=labels
            )
            image = transformed['image']
            boxes = np.array(transformed['bboxes'], dtype=np.float32)
            labels = np.array(transformed['labels'], dtype=np.int64)
            # 🔥 CORRECCIÓN CRÍTICA: Convertir [xmin, ymin, xmax, ymax] -> [ymin, xmin, ymax, xmax]
            if len(boxes) > 0:
                boxes = boxes[:, [1, 0, 3, 2]]
        
        if len(boxes) == 0:
            boxes = np.zeros((0, 4), dtype=np.float32)
            labels = np.zeros((0,), dtype=np.int64)
        
        target = {
            'bbox': torch.tensor(boxes, dtype=torch.float32),
            'cls': torch.tensor(labels, dtype=torch.int64),
            'img_scale': torch.tensor([1.0], dtype=torch.float32),
            'img_size': torch.tensor([image.shape[1], image.shape[2]], dtype=torch.int64)
        }
        
        return image, target

print("✅ Clase YOLODataset definida")

In [ ]:
# ── Augmentations ──────────────────────────────────────────────────────────────
def get_train_transforms(img_size=IMG_SIZE):
    return A.Compose([
        A.LongestMaxSize(max_size=img_size),
        A.PadIfNeeded(min_height=img_size, min_width=img_size, border_mode=0, value=0),
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.3),
        A.Blur(blur_limit=3, p=0.2),
        A.GaussNoise(p=0.2),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))

def get_val_transforms(img_size=IMG_SIZE):
    return A.Compose([
        A.LongestMaxSize(max_size=img_size),
        A.PadIfNeeded(min_height=img_size, min_width=img_size, border_mode=0, value=0),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))

def collate_fn(batch):
    images, targets = zip(*batch)
    images = torch.stack(images)
    return images, targets

print("✅ Funciones de augmentation definidas")

In [ ]:
# ── YAML helpers ───────────────────────────────────────────────────────────────
def create_combined_yaml() -> str:
    base = os.path.abspath(EC_BASE)
    ec1_train = os.path.join(base, "license-plates-ec-1", "train", "images").replace("\\", "/")
    ec1_valid = os.path.join(base, "license-plates-ec-1", "valid", "images").replace("\\", "/")
    ec1_test  = os.path.join(base, "license-plates-ec-1", "test",  "images").replace("\\", "/")
    ec2_train = os.path.join(base, "license-plates-ec-2", "train", "images").replace("\\", "/")
    ec4_train = os.path.join(base, "license-plates-ec-4", "train", "images").replace("\\", "/")

    data = {
        "train": [ec1_train, ec2_train, ec4_train],
        "val":   ec1_valid,
        "test":  ec1_test,
        "nc":    1,
        "names": ["license plate"],
    }

    path = os.path.join(base, "data_combined.yaml")
    with open(path, "w") as f:
        yaml.dump(data, f, default_flow_style=False, allow_unicode=True)
    return path

def create_combined_all_yaml() -> str:
    base    = os.path.abspath(EC_BASE)
    global_ = os.path.abspath("datasets/license-plates")
    
    gl_train  = os.path.join(global_, "train", "images").replace("\\", "/")
    gl_valid  = os.path.join(global_, "valid", "images").replace("\\", "/")
    ec1_train = os.path.join(base, "license-plates-ec-1", "train", "images").replace("\\", "/")
    ec2_train = os.path.join(base, "license-plates-ec-2", "train", "images").replace("\\", "/")
    ec4_train = os.path.join(base, "license-plates-ec-4", "train", "images").replace("\\", "/")

    data = {
        "train": [gl_train, ec1_train, ec2_train, ec4_train],
        "val":   gl_valid,
        "test":  os.path.join(global_, "test", "images").replace("\\", "/"),
        "nc":    1,
        "names": ["license plate"],
    }

    path = os.path.join(base, "data_combined_all.yaml")
    with open(path, "w") as f:
        yaml.dump(data, f, default_flow_style=False, allow_unicode=True)
    return path

def load_yaml_paths(dataset_key: str) -> Tuple[List[str], str, str]:
    if dataset_key == "combined":
        yaml_path = create_combined_yaml()
    elif dataset_key == "combined_all":
        yaml_path = create_combined_all_yaml()
    else:
        yaml_path = DATASETS[dataset_key]["yaml"]
    
    with open(yaml_path, 'r') as f:
        data = yaml.safe_load(f)
    
    train = data['train'] if isinstance(data['train'], list) else [data['train']]
    val   = data['val']
    test  = data.get('test', val)
    
    return train, val, test

print("✅ YAML helpers definidos")

In [ ]:
# ── Modelo EfficientDet ────────────────────────────────────────────────────────
def create_model(num_classes=NUM_CLASSES, pretrained=True):
    config = get_efficientdet_config(MODEL_NAME)
    config.num_classes = num_classes
    config.image_size = IMG_SIZE
    
    model = EfficientDet(config, pretrained_backbone=pretrained)
    model.class_net = HeadNet(
        config,
        num_outputs=num_classes,
    )
    
    model = DetBenchTrain(model, config)
    return model

print("✅ Función create_model definida")

In [ ]:
# ── Training functions ─────────────────────────────────────────────────────────
def train_one_epoch(model, dataloader, optimizer, device, epoch, scaler=None):
    model.train()
    running_loss = 0.0
    
    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for images, targets in pbar:
        images = images.to(device)
        
        batch_targets = {}
        for key in ['bbox', 'cls', 'img_scale', 'img_size']:
            batch_targets[key] = torch.stack([t[key] for t in targets]).to(device)
        
        optimizer.zero_grad()
        
        if USE_AMP and scaler is not None:
            with autocast():
                loss_dict = model(images, batch_targets)
                loss = loss_dict['loss']
            
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss_dict = model(images, batch_targets)
            loss = loss_dict['loss']
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        
        running_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return running_loss / len(dataloader)

@torch.no_grad()
def validate(model, dataloader, device):
    model.eval()
    running_loss = 0.0
    
    for images, targets in tqdm(dataloader, desc="Validating"):
        images = images.to(device)
        
        batch_targets = {}
        for key in ['bbox', 'cls', 'img_scale', 'img_size']:
            batch_targets[key] = torch.stack([t[key] for t in targets]).to(device)
        
        if USE_AMP and device == "cuda":
            with autocast():
                loss_dict = model(images, batch_targets)
                loss = loss_dict['loss']
        else:
            loss_dict = model(images, batch_targets)
            loss = loss_dict['loss']
        
        running_loss += loss.item()
    
    return running_loss / len(dataloader)

print("✅ Funciones de entrenamiento definidas")

In [ ]:
# ── Exportación ONNX ───────────────────────────────────────────────────────────
def export_to_onnx(model_path: str, output_path: str, img_size=IMG_SIZE):
    import onnx
    
    print(f"\n[ONNX Export] Exportando {model_path} → {output_path}")
    
    checkpoint = torch.load(model_path, map_location=DEVICE)
    model = create_model(num_classes=NUM_CLASSES, pretrained=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(DEVICE)
    model.eval()
    
    dummy_input = torch.randn(1, 3, img_size, img_size).to(DEVICE)
    
    torch.onnx.export(
        model.model,
        dummy_input,
        output_path,
        export_params=True,
        opset_version=12,
        do_constant_folding=True,
        input_names=['images'],
        output_names=['boxes', 'scores', 'classes'],
        dynamic_axes={
            'images': {0: 'batch_size'},
            'boxes': {0: 'batch_size'},
            'scores': {0: 'batch_size'},
            'classes': {0: 'batch_size'}
        }
    )
    
    onnx_model = onnx.load(output_path)
    onnx.checker.check_model(onnx_model)
    
    print(f"  ✅ ONNX exportado: {os.path.getsize(output_path) / (1024**2):.2f} MB")

print("✅ Función export_to_onnx definida")

In [ ]:
# ── Función de entrenamiento principal ────────────────────────────────────────
def train(dataset_key: str):
    cfg = DATASETS[dataset_key]
    model_name = cfg["name"]
    
    print("\n" + "=" * 60)
    print(f"  Entrenando: {dataset_key.upper()}")
    print(f"  Modelo:     {model_name}")
    print(f"  Arquitectura: {MODEL_NAME}")
    print(f"  Dispositivo: {DEVICE}")
    print(f"  Mixed Precision: {'✅ ENABLED' if USE_AMP else '❌ DISABLED'}")
    print(f"  Batch size: {BATCH_SIZE}")
    print("=" * 60)
    
    train_dirs, val_dir, test_dir = load_yaml_paths(dataset_key)
    
    train_dataset = YOLODataset(train_dirs, transform=get_train_transforms())
    val_dataset   = YOLODataset(val_dir, transform=get_val_transforms())
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        collate_fn=collate_fn,
        pin_memory=True if DEVICE == "cuda" else False
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        collate_fn=collate_fn,
        pin_memory=True if DEVICE == "cuda" else False
    )
    
    model = create_model(num_classes=NUM_CLASSES, pretrained=True)
    model = model.to(DEVICE)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler = GradScaler() if USE_AMP and DEVICE == "cuda" else None
    
    save_dir = Path(f"runs/detect/{model_name}")
    save_dir.mkdir(parents=True, exist_ok=True)
    weights_dir = save_dir / "weights"
    weights_dir.mkdir(exist_ok=True)
    
    # ── Mecanismo de Auto-Resiliencia (Resume) ──
    start_epoch = 0
    best_val_loss = float('inf')
    last_checkpoint_path = weights_dir / "last.pt"
    
    if last_checkpoint_path.exists():
        print(f"🔄 Encontrado checkpoint de sesión anterior: {last_checkpoint_path}")
        print("   Cargando pesos, optimizador y época para continuar...")
        checkpoint = torch.load(last_checkpoint_path, map_location=DEVICE)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        best_val_loss = checkpoint.get('val_loss', float('inf'))
        
        # Ajustar el scheduler al estado de época actual
        for _ in range(start_epoch):
            scheduler.step()
        print(f"✅ Reanudación exitosa desde Época {start_epoch + 1}")
    
    patience = 10
    patience_counter = 0
    for epoch in range(start_epoch, EPOCHS):
        print(f"\n[Epoch {epoch+1}/{EPOCHS}]")
        
        train_loss = train_one_epoch(model, train_loader, optimizer, DEVICE, epoch, scaler)
        val_loss = validate(model, val_loader, DEVICE)
        
        scheduler.step()
        
        print(f"  Train Loss: {train_loss:.4f}")
        print(f"  Val Loss:   {val_loss:.4f}")
        print(f"  LR:         {optimizer.param_groups[0]['lr']:.6f}")
        
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
        }, weights_dir / "last.pt")
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_loss': train_loss,
                'val_loss': val_loss,
            }, weights_dir / "best.pt")
            print(f"  ✅ Mejor modelo guardado (val_loss: {val_loss:.4f})")
        else:
            patience_counter += 1
        
        if patience_counter >= patience:
            print(f"\n  Early stopping después de {epoch+1} épocas")
            break
    
    best_path = str(weights_dir / "best.pt")
    print(f"\n  ✅ Entrenamiento completado")
    print(f"  ✅ Mejor modelo: {best_path}")
    
    if EXPORT_ONNX:
        onnx_path = str(weights_dir / "best.onnx")
        export_to_onnx(best_path, onnx_path, img_size=IMG_SIZE)
    
    return best_path

print("✅ Función train definida")

## 🚀 Ejecutar entrenamiento

**Selecciona el dataset que quieres entrenar:**

In [ ]:
# Optimizaciones CUDA
if DEVICE == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.enabled = True
    torch.cuda.empty_cache()
    print("✅ Optimizaciones CUDA activadas")

# ═══════════════════════════════════════════════════════════════════════════════
# SELECCIONA UNO (descomenta la línea que quieras usar):
# ═══════════════════════════════════════════════════════════════════════════════

# dataset_to_train = "global"         # 10,125 imágenes (dataset completo)
# dataset_to_train = "ecuador"        # ec-1 (144 imágenes)
# dataset_to_train = "ecuador2"       # ec-2 (90 imágenes)
# dataset_to_train = "ecuador4"       # ec-4 (375 imágenes)
# dataset_to_train = "combined"       # ec-1 + ec-2 + ec-4 combinados
dataset_to_train = "combined_all"     # global + ec-1 + ec-2 + ec-4 (RECOMENDADO)

# ═══════════════════════════════════════════════════════════════════════════════

print(f"\n🎯 Dataset seleccionado: {dataset_to_train.upper()}")
print(f"🚀 Iniciando entrenamiento...\n")

# Entrenar
best_model_path = train(dataset_to_train)

print(f"\n{'='*60}")
print(f"  ✅ ENTRENAMIENTO COMPLETADO")
print(f"  📦 Modelo PyTorch: {best_model_path}")
print(f"  📦 Modelo ONNX: {best_model_path.replace('.pt', '.onnx')}")
print(f"{'='*60}")

## 📊 Visualizar pérdidas de entrenamiento

In [ ]:
# Crear gráfica de pérdidas si se guardaron logs
# (Nota: Necesitarías guardar las pérdidas en cada época para visualizarlas)

print("📊 Ver resultados en la carpeta runs/detect/")
!ls -lh runs/detect/{DATASETS[dataset_to_train]['name']}/weights/

## 💾 Descargar modelos entrenados

In [ ]:
from google.colab import files
import shutil

model_name = DATASETS[dataset_to_train]['name']

# Comprimir carpeta completa de resultados
print("📦 Comprimiendo resultados...")
shutil.make_archive(f"{model_name}", 'zip', f"runs/detect/{model_name}")

print(f"\n⬇️  Descargando {model_name}.zip...")
files.download(f"{model_name}.zip")

print("\n⬇️  Descargando pesos individuales...")
files.download(f"runs/detect/{model_name}/weights/best.pt")
files.download(f"runs/detect/{model_name}/weights/best.onnx")

print("\n✅ Descargas completadas")

## 💡 Guardar en Google Drive (persistencia)

**Importante:** Ejecuta esto para no perder tus modelos cuando se cierre la sesión de Colab.

In [ ]:
# Crear carpeta en Drive si no existe
!mkdir -p /content/drive/MyDrive/efficientdet_training_results

# Copiar resultados completos a Drive
!cp -r runs/detect/* /content/drive/MyDrive/efficientdet_training_results/

print("\n✅ Resultados guardados en Google Drive")
print("📂 Ubicación: /content/drive/MyDrive/efficientdet_training_results/")
print("\n💡 Tus modelos están seguros y podrás accederlos desde cualquier sesión de Colab")

## 🔍 Inferencia rápida con imagen de prueba

In [ ]:
# Subir imagen de prueba
from google.colab import files
import matplotlib.pyplot as plt

print("📤 Selecciona una imagen para probar el modelo...")
uploaded = files.upload()
test_image_path = list(uploaded.keys())[0]

# Cargar modelo
print("\n🔄 Cargando modelo...")
checkpoint = torch.load(best_model_path, map_location=DEVICE)
model = create_model(num_classes=NUM_CLASSES, pretrained=False)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(DEVICE)
model.eval()

# Leer y preprocesar imagen
image = cv2.imread(test_image_path)
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
h, w = image_rgb.shape[:2]

transform = get_val_transforms()
transformed = transform(image=image_rgb, bboxes=[], labels=[])
input_tensor = transformed['image'].unsqueeze(0).to(DEVICE)

# Inferencia
print("🔍 Detectando placas...")
with torch.no_grad():
    if USE_AMP:
        with autocast():
            outputs = model.model(input_tensor)
    else:
        outputs = model.model(input_tensor)

print(f"\n✅ Inferencia completada")
print(f"📊 Outputs: {type(outputs)}")

# Visualizar imagen
plt.figure(figsize=(12, 8))
plt.imshow(image_rgb)
plt.title(f"Imagen de prueba: {test_image_path}")
plt.axis('off')
plt.show()

print("\n💡 Nota: Para visualizar las detecciones completas, necesitas implementar")
print("   el postprocesamiento de las salidas del modelo (NMS, threshold, etc.)")

## 📈 Benchmark PyTorch vs ONNX

Compara la velocidad de inferencia entre PyTorch y ONNX.

In [ ]:
import time
import onnxruntime as ort

def benchmark_pytorch(model_path, iterations=100):
    checkpoint = torch.load(model_path, map_location=DEVICE)
    model = create_model(num_classes=NUM_CLASSES, pretrained=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(DEVICE).eval()
    
    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
    
    # Warmup
    with torch.no_grad():
        for _ in range(10):
            _ = model.model(dummy)
    
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    
    start = time.time()
    with torch.no_grad():
        for _ in range(iterations):
            _ = model.model(dummy)
    
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    
    return (time.time() - start) / iterations * 1000

def benchmark_onnx(onnx_path, iterations=100):
    providers = ['CUDAExecutionProvider'] if DEVICE == "cuda" else ['CPUExecutionProvider']
    session = ort.InferenceSession(onnx_path, providers=providers)
    dummy = np.random.randn(1, 3, IMG_SIZE, IMG_SIZE).astype(np.float32)
    input_name = session.get_inputs()[0].name
    
    # Warmup
    for _ in range(10):
        _ = session.run(None, {input_name: dummy})
    
    start = time.time()
    for _ in range(iterations):
        _ = session.run(None, {input_name: dummy})
    
    return (time.time() - start) / iterations * 1000

# Ejecutar benchmark
print("\n" + "="*60)
print("  BENCHMARK: PyTorch vs ONNX (100 iteraciones)")
print("="*60)

print("\n⏱️  Benchmarking PyTorch...")
pytorch_time = benchmark_pytorch(best_model_path)

print("⏱️  Benchmarking ONNX...")
onnx_time = benchmark_onnx(best_model_path.replace('.pt', '.onnx'))

speedup = pytorch_time / onnx_time

print(f"\n{'='*60}")
print(f"  RESULTADOS:")
print(f"{'='*60}")
print(f"  PyTorch:  {pytorch_time:.2f} ms/imagen ({1000/pytorch_time:.1f} FPS)")
print(f"  ONNX:     {onnx_time:.2f} ms/imagen ({1000/onnx_time:.1f} FPS)")
print(f"  Speedup:  {speedup:.2f}x más rápido")
print(f"{'='*60}")

if speedup > 1.2:
    print(f"\n✅ ONNX es significativamente más rápido - usa este modelo en producción")
else:
    print(f"\n💡 Diferencia menor - puedes usar cualquiera de los dos")

## 🎓 Resumen y próximos pasos

### ✅ Lo que has logrado:
1. Entrenado EfficientDet-D2 con GPU gratuita de Colab
2. Exportado modelo a formato ONNX optimizado
3. Guardado modelos en Google Drive para persistencia
4. Comparado velocidad PyTorch vs ONNX

### 🚀 Próximos pasos recomendados:

1. **Probar el modelo en más imágenes**
   - Sube varias imágenes y visualiza detecciones
   - Calcula métricas (mAP, precision, recall)

2. **Fine-tuning**
   - Si el modelo no funciona bien, ajusta hiperparámetros:
     - Aumenta `EPOCHS` a 150-200
     - Prueba diferentes `BATCH_SIZE` (4, 8, 12)
     - Modifica `LR` (5e-5, 1e-4, 5e-4)

3. **Deployment**
   - Usa el modelo ONNX en tu aplicación FastAPI
   - Implementa pipeline completo: detección → OCR → validación

4. **Comparar con YOLO**
   - Entrena YOLOv8 con los mismos datos
   - Compara velocidad vs precisión

### 📚 Recursos útiles:
- [EfficientDet Paper](https://arxiv.org/abs/1911.09070)
- [ONNX Runtime Docs](https://onnxruntime.ai/)
- [Colab Pro](https://colab.research.google.com/signup) (GPUs más potentes)

---

**¿Preguntas?** Revisa la documentación o ajusta los parámetros según tus necesidades.